In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import random
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_clean_dataset(base_dir: str, dataset_name: str) -> pd.DataFrame:
    csv_path = os.path.join(base_dir, "clean", f"{dataset_name}_clean.csv")
    return pd.read_csv(csv_path).reset_index(drop=True)


def encode_texts_for_dataset(df, tokenizer, transformer, device, max_length=128, batch_size=32):
    texts = df["text"].astype(str).tolist()
    all_embeddings = []
    transformer.eval()
    with torch.no_grad():
        batches = int(np.ceil(len(texts) / batch_size))
        for i in tqdm(range(batches), desc="  Encoding texts"):
            batch_texts = texts[i * batch_size:(i + 1) * batch_size]
            enc = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}
            outputs = transformer(**enc)
            cls_emb = outputs.last_hidden_state[:, 0, :]
            all_embeddings.append(cls_emb.cpu())
    return torch.cat(all_embeddings, dim=0)


def prepare_all_datasets(base_dir, model_name, max_length, batch_size, device, dataset_names):
    print(f"Using device: {device}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    transformer = AutoModel.from_pretrained(model_name).to(device)
    all_data = {}
    for name in dataset_names:
        print(f"\n=== Preparing dataset: {name} ===")
        df = load_clean_dataset(base_dir, name)
        print(f"  Num documents: {len(df)}")
        embeddings = encode_texts_for_dataset(df, tokenizer, transformer, device, max_length, batch_size)
        labels = torch.tensor(df["label"].values, dtype=torch.long)
        split = df["split"].values
        train_mask = torch.tensor(split == "train", dtype=torch.bool)
        val_mask = torch.tensor(split == "val", dtype=torch.bool)
        test_mask = torch.tensor(split == "test", dtype=torch.bool)
        all_data[name] = {"emb": embeddings, "labels": labels, "train_mask": train_mask,
                           "val_mask": val_mask, "test_mask": test_mask}
    return all_data


class TextOnlyClassifier(nn.Module):
    def __init__(self, emb_dim, hidden=256, num_classes=2, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(emb_dim, hidden)
        self.fc2 = nn.Linear(hidden, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        h = self.dropout(h)
        return self.fc2(h)


def resample_train_mask(labels, orig_train_mask, target_fake_ratio, seed):
    rng = np.random.RandomState(seed)
    train_idx = torch.nonzero(orig_train_mask, as_tuple=True)[0].numpy()
    train_labels = labels[orig_train_mask].numpy()
    fake_idx = train_idx[train_labels == 1]
    real_idx = train_idx[train_labels == 0]
    Fk, R = len(fake_idx), len(real_idx)
    p = target_fake_ratio
    new_fake_a = Fk
    new_real_a = round(Fk * (1 - p) / p) if p > 0 else R
    new_real_b = R
    new_fake_b = round(R * p / (1 - p)) if p < 1 else Fk
    if new_real_a <= R:
        new_fake, new_real = new_fake_a, new_real_a
    else:
        new_fake, new_real = new_fake_b, new_real_b
    new_fake = max(1, min(new_fake, Fk))
    new_real = max(1, min(new_real, R))
    chosen_fake = rng.choice(fake_idx, size=new_fake, replace=False)
    chosen_real = rng.choice(real_idx, size=new_real, replace=False)
    chosen = np.concatenate([chosen_fake, chosen_real])
    new_mask = torch.zeros_like(orig_train_mask)
    new_mask[chosen] = True
    return new_mask, len(chosen)


def lr_lambda(epoch, total_epochs, warmup_frac=0.1):
    warmup_epochs = max(1, int(total_epochs * warmup_frac))
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
    return 0.5 * (1 + np.cos(np.pi * progress))


def train_one_run(
    src_name, all_data, device, target_fake_ratio, seed, batch_size_mode,
    epochs=30, lr=1e-3, weight_decay=1e-4, hidden=256, dropout=0.3,
    grad_clip_norm=1.0, collapse_epoch_start=1, collapse_threshold=0.98,
):
    """
    batch_size_mode: "full" for full-batch (one gradient step per epoch, matching
    every other experiment in this paper), or an integer mini-batch size (multiple
    shuffled gradient steps per epoch). LR schedule is stepped once per EPOCH in both
    cases (not per mini-batch step), so warmup/decay timing stays directly comparable
    to every other experiment in this paper regardless of batch size.
    UNWEIGHTED loss throughout -- this experiment specifically tests whether collapse,
    established in the unweighted/imbalanced regime, depends on full-batch training.
    """
    set_seed(seed)

    data_src = all_data[src_name]
    x_src = data_src["emb"].to(device)
    y_src = data_src["labels"].to(device)
    val_mask = data_src["val_mask"].to(device)
    test_mask = data_src["test_mask"].to(device)

    train_mask, n_train = resample_train_mask(data_src["labels"], data_src["train_mask"], target_fake_ratio, seed)
    train_mask = train_mask.to(device)
    train_idx_all = torch.nonzero(train_mask, as_tuple=True)[0]

    emb_dim = x_src.size(1)
    model = TextOnlyClassifier(emb_dim, hidden, 2, dropout).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda ep: lr_lambda(ep, epochs))
    criterion = nn.CrossEntropyLoss()

    rng = np.random.RandomState(seed + 10000)  # separate RNG stream for batch shuffling
    epoch_first_collapse = None
    val_f1_curve, val_pred_fake_frac_curve = [], []
    n_steps_per_epoch = []

    for epoch in range(epochs):
        model.train()
        if batch_size_mode == "full":
            optimizer.zero_grad()
            logits = model(x_src)
            loss = criterion(logits[train_mask], y_src[train_mask])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
            optimizer.step()
            n_steps_per_epoch.append(1)
        else:
            bs = int(batch_size_mode)
            perm = train_idx_all[torch.from_numpy(rng.permutation(len(train_idx_all)))]
            steps = 0
            for i in range(0, len(perm), bs):
                batch_idx = perm[i:i + bs]
                optimizer.zero_grad()
                logits = model(x_src[batch_idx])
                loss = criterion(logits, y_src[batch_idx])
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
                optimizer.step()
                steps += 1
            n_steps_per_epoch.append(steps)
        scheduler.step()  # once per epoch regardless of batch size, for comparability

        model.eval()
        with torch.no_grad():
            val_logits = model(x_src)[val_mask]
            val_pred = val_logits.argmax(dim=-1)
            val_f1 = f1_score(y_src[val_mask].cpu().numpy(), val_pred.cpu().numpy(), average="macro")
            pred_fake_frac = val_pred.float().mean().item()
        val_f1_curve.append(val_f1)
        val_pred_fake_frac_curve.append(pred_fake_frac)

        is_collapsed_now = (pred_fake_frac >= collapse_threshold) or (pred_fake_frac <= (1 - collapse_threshold))
        if is_collapsed_now and epoch >= collapse_epoch_start and epoch_first_collapse is None:
            epoch_first_collapse = epoch

    model.eval()
    with torch.no_grad():
        test_logits = model(x_src)[test_mask]
        test_pred = test_logits.argmax(dim=-1)
        test_labels = y_src[test_mask]
        test_acc = accuracy_score(test_labels.cpu().numpy(), test_pred.cpu().numpy())
        test_f1 = f1_score(test_labels.cpu().numpy(), test_pred.cpu().numpy(), average="macro")
        test_pred_fake_frac = test_pred.float().mean().item()

    final_pred_fake_frac = val_pred_fake_frac_curve[-1]
    collapsed_final = (final_pred_fake_frac >= collapse_threshold) or (final_pred_fake_frac <= (1 - collapse_threshold))

    return {
        "source": src_name, "batch_size_mode": str(batch_size_mode), "n_train": n_train,
        "steps_per_epoch": n_steps_per_epoch[0] if n_steps_per_epoch else None,
        "seed": seed, "target_fake_ratio": target_fake_ratio,
        "epoch_first_collapse": epoch_first_collapse if epoch_first_collapse is not None else -1,
        "collapsed_final": bool(collapsed_final),
        "test_macro_f1": test_f1, "test_acc": test_acc, "test_pred_fake_frac": test_pred_fake_frac,
        "final_val_macro_f1": val_f1_curve[-1],
    }


def run_batch_size_experiment(
    base_dir, sources=["coaid", "fakenewsnet"], target_fake_ratio=0.80,
    batch_size_modes=["full", 128, 32, 8], seeds=list(range(5)),
    model_name="roberta-base", epochs=30, max_length=128, dropout=0.3,
):
    device = get_device()
    all_data = prepare_all_datasets(base_dir, model_name, max_length, 32, device, sources)

    out_dir = os.path.join(base_dir, "batch_size_results")
    os.makedirs(out_dir, exist_ok=True)

    rows = []
    total = len(sources) * len(batch_size_modes) * len(seeds)
    counter = 0
    print(f"\n=== Full-batch vs mini-batch experiment: {total} runs "
          f"(fixed fake-ratio={target_fake_ratio}, unweighted, T-Base only) ===")
    for src in sources:
        for bsm in batch_size_modes:
            for seed in seeds:
                counter += 1
                result = train_one_run(src, all_data, device, target_fake_ratio, seed, bsm,
                                        epochs=epochs, dropout=dropout)
                print(f"[{counter}/{total}] {src} batch={bsm} seed={seed} steps/epoch={result['steps_per_epoch']} "
                      f"-> first_collapse_epoch={result['epoch_first_collapse']} "
                      f"collapsed_final={result['collapsed_final']} test_F1={result['test_macro_f1']:.4f}")
                rows.append(result)

    df = pd.DataFrame(rows)
    out_path = os.path.join(out_dir, "batch_size_summary.csv")
    df.to_csv(out_path, index=False)
    print(f"\nSaved to: {out_path}")

    print("\n=== Collapse rate and mean first-collapse epoch by batch size mode ===")
    summary = df.groupby(["source", "batch_size_mode"]).agg(
        steps_per_epoch=("steps_per_epoch", "first"),
        collapse_rate=("collapsed_final", "mean"),
        mean_first_collapse_epoch=("epoch_first_collapse", lambda x: x[x >= 0].mean() if (x >= 0).any() else np.nan),
        n=("seed", "count"),
    ).reset_index()
    print(summary.to_string(index=False))

    return df


run_batch_size_experiment(
    base_dir="drive/MyDrive/CSML/data",
    sources=["coaid", "fakenewsnet"],
    target_fake_ratio=0.80,
    batch_size_modes=["full", 128, 32, 8],
    seeds=list(range(5)),
    model_name="roberta-base",
    epochs=30,
    max_length=128,
    dropout=0.3,
)
